## Assignment 1: Supervised Machine Learning

Welcome to the first assignment of CS 541! This assignment prepares you with some useful tools that are widely used in NLP. This assignment must be done individually.

After this assignment, you should be able to:  
1. Load a dataset from huggingface's dataset library, and do some exploratory analyses.  
2. Use scikit-learn to build and train a feature-based model.  
3. Use pytorch to build and train a feature-based model.  
4. Use Optuna to automatically search for hyperparameters.  

In CS541, any work generated by an AI shouldn't be included without declaration. If you include material generated by an AI, the level of AI use should be properly documented, and the actual tool should be noted (e.g., "I used Codex to proofread the codes and draft the analysis"). 

### 1. Load the dataset (5')
First, we are going to load the datasets from huggingface's `datasets` library.
Do some exploratory analysis on the dataset.  
1.1 Print out one example in the dataset. Briefly comment on what it contains.  
1.2 For each of the train, validation, and test set, compute the following statistics: 
- The number of data samples with each class label.  
- The mean and std of the sentence lengths (in words) of each `question`.  

1.3 Vectorize the validation set of the dataset, following the approaches specified in the train set example.

In [3]:
import pandas as pd 
import numpy as np 
from datasets import load_dataset  # huggingface datasets

ds = load_dataset("stanfordnlp/sst2")

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [5]:
# TODO -- Print out one example in the dataset. Briefly comment on what it contains.
example = ds["train"][0]
print(example)
print("The example contains a sentence, its SST-2 sentiment label, and its dataset index.")

{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0}
The example contains a sentence, its SST-2 sentiment label, and its dataset index.


In [6]:
train_data = pd.DataFrame(ds["train"])
X_train_text = train_data["sentence"]
Y_train = train_data["label"]

val_data = pd.DataFrame(ds["validation"])
X_val_text = val_data["sentence"]
Y_val = val_data["label"]

test_data = pd.DataFrame(ds["test"])
X_test_text = test_data["sentence"]
Y_test = test_data["label"]

In [7]:
# TODO -- compute the exploratory statistics
for split_name in ["train", "validation", "test"]:
    df = pd.DataFrame(ds[split_name])
    sentence_lengths = df["sentence"].str.split().str.len()
    print("\n===== {} =====".format(split_name))
    print("Class counts:")
    print(df["label"].value_counts().sort_index())
    print("Mean sentence length (words): {:.4f}".format(sentence_lengths.mean()))
    print("Std sentence length (words): {:.4f}".format(sentence_lengths.std()))


===== train =====
Class counts:
label
0    29780
1    37569
Name: count, dtype: int64
Mean sentence length (words): 9.4096
Std sentence length (words): 8.0738

===== validation =====
Class counts:
label
0    428
1    444
Name: count, dtype: int64
Mean sentence length (words): 19.5482
Std sentence length (words): 8.7639

===== test =====
Class counts:
label
-1    1821
Name: count, dtype: int64
Mean sentence length (words): 19.2339
Std sentence length (words): 8.9224


Next we are going to vectorize the texts using TfidfVectorizer, then compute the Tf-idf features.   

In [8]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer 
from sklearn.neural_network import MLPClassifier 

counter = CountVectorizer(min_df=10, max_df=20) 
counter.fit(X_train_text)
print("Vocabulary size:", len(counter.vocabulary_))
X_train_counts = counter.transform(X_train_text)
print(X_train_counts.shape) 
count2tfidf = TfidfTransformer(use_idf=True).fit(X_train_counts)
X_train = count2tfidf.transform(X_train_counts).toarray()
print(X_train.shape)

Vocabulary size: 3120
(67349, 3120)
(67349, 3120)


In [9]:
# TODO - Use the counter to convert X_val_text to occurrence vectors
# Note: don't create a new CountVectorizer, as we want to compute the vocabulary only on the train set
X_val_counts = counter.transform(X_val_text)

# TODO - use count2tfidf to transform the counts into Tfidf features
# Note: don't create a new TfidfTransformer
X_val = count2tfidf.transform(X_val_counts).toarray()

print("Validation count shape:", X_val_counts.shape)
print("Validation TF-IDF shape:", X_val.shape)

Validation count shape: (872, 3120)
Validation TF-IDF shape: (872, 3120)


### 2. Train scikit-learn models (10')
Train a two-layer MLPClassifier using `random_state=0`. Manually tune the hyperparameters on the validation set. Report the procedure of hyperparameter tuning. Specifically: report the hyperparameters you have tried, and their results.  

After you are satisfied with the validation set performances, report the validation set performance. Use this set of hyperparameters and repeat the model training procedure for five times using `random_state` as 1, 2, 3, 31, 42 respectively. Record the five accuracy numbers.

In [10]:
# Starter
def train_sklearn_model(X_train, Y_train, X_val, Y_val):
    # Manual tuning; random_state=0 is required for this stage.
    tuning_configs = [
        {"hidden_layer_sizes": (50,), "learning_rate_init": 0.001, "batch_size": 128, "max_iter": 15},
        {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001, "batch_size": 128, "max_iter": 15},
        {"hidden_layer_sizes": (100,), "learning_rate_init": 0.003, "batch_size": 128, "max_iter": 15},
        {"hidden_layer_sizes": (200,), "learning_rate_init": 0.001, "batch_size": 128, "max_iter": 15},
    ]
    tuning_results = []
    print("Manual hyperparameter tuning (random_state=0)")
    for config in tuning_configs:
        model = MLPClassifier(random_state=0, **config)
        model.fit(X_train, Y_train)
        acc = model.score(X_val, Y_val)
        tuning_results.append({**config, "validation_accuracy": acc})
        print(config, "->", acc)
    tuning_df = pd.DataFrame(tuning_results)
    print("\nTuning results:")
    print(tuning_df.to_string(index=False))
    best = tuning_df.loc[tuning_df["validation_accuracy"].idxmax()]
    best_params = {
        "hidden_layer_sizes": best["hidden_layer_sizes"],
        "learning_rate_init": float(best["learning_rate_init"]),
        "batch_size": int(best["batch_size"]),
        "max_iter": int(best["max_iter"]),
    }
    print("\nSelected hyperparameters:", best_params)
    final_model = MLPClassifier(random_state=0, **best_params)
    final_model.fit(X_train, Y_train)
    final_val_accuracy = final_model.score(X_val, Y_val)
    print("Final validation accuracy (random_state=0):", final_val_accuracy)
    global sklearn_seed_accuracies
    sklearn_seed_accuracies = []
    print("\nFive required random seeds:")
    for seed in [1, 2, 3, 31, 42]:
        model = MLPClassifier(random_state=seed, **best_params)
        model.fit(X_train, Y_train)
        acc = model.score(X_val, Y_val)
        sklearn_seed_accuracies.append(acc)
        print("random_state = {}, validation accuracy = {:.6f}".format(seed, acc))
    return final_val_accuracy

train_sklearn_model(X_train, Y_train, X_val, Y_val)

Manual hyperparameter tuning (random_state=0)


c:\Users\manda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


{'hidden_layer_sizes': (50,), 'learning_rate_init': 0.001, 'batch_size': 128, 'max_iter': 15} -> 0.569954128440367


c:\Users\manda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001, 'batch_size': 128, 'max_iter': 15} -> 0.5779816513761468


...

random_state = 3, validation accuracy = 0.579128


c:\Users\manda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


random_state = 31, validation accuracy = 0.583716
random_state = 42, validation accuracy = 0.586009


c:\Users\manda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


0.5802752293577982

### 3. Train a pytorch model (10')
Here you will repeat the training of a two-layer fully-connected neural network using pytorch. Following are some specifications that may be helpful:  
- For each of the train and validation set, specify a dataloader, preferrably using `torch.utils.data.DataLoader`.  
- Use an optimizer of your choice. Adam, AdamW and SGD are popular choices.  
- Designate a number, `train_epochs`, as the number of passes through the dataset during training. Each pass through the training dataset is called an epoch.  
  - During the epoch, there may be many steps. In each step, load a batch of data from the dataloader. Compute the loss. Do a `backward()` pass to compute the gradients. Call a `step()` from the optimizer to update the model's parameters. Then zero out the gradients.
- At the end of each epoch, go through a validation run. Do *not* optimize the model during the validation run. Compute the accuracy of the model on this validation run, and print it out.

Tune the hyperparameters on the validation set. Report the hyperparameters you have tried, and their results. 

After you are satisfied with the validation set performances, record the set of hyperparameters. Use this set of hyperparameters, and repeat the model training procedure for five times using 1, 2, 3, 31, 42 as random seeds respectively. You can use `torch.manual_seed()` to set the random seeds. Record the five accuracy numbers.

In [11]:
# Starter
import torch
import torch.nn as nn
from collections import OrderedDict

class MLP(nn.Module):
    def __init__(self, all_layer_sizes):
        super().__init__()
        layers = OrderedDict()
        for i in range(len(all_layer_sizes) - 1):
            layers["linear_{}".format(i)] = nn.Linear(all_layer_sizes[i], all_layer_sizes[i + 1])
            if i < len(all_layer_sizes) - 2:
                layers["relu_{}".format(i)] = nn.ReLU()
        self.net = nn.Sequential(layers)

    def forward(self, X):
        return self.net(X)

def my_collate_function(batch):
    batch_X, batch_Y = [], []
    for item in batch:
        batch_X.append(item[0])
        batch_Y.append(item[1])
    return torch.tensor(batch_X).float(), torch.tensor(batch_Y).long()

def prepare_zipped_XY(X, Y):
    zipped = []
    for i in range(len(X)):
        zipped.append((X[i], Y[i]))
    return zipped

def train_pytorch_model(X_train, Y_train, X_val, Y_val):
    # Define the manual seed
    torch.manual_seed(1)

    # Define the hyperparameters
    train_epochs = 10
    batch_size = 128
    learning_rate = 0.001
    hidden_sizes = [100]

    # Set up the model, optimizer, and dataloader
    all_layer_sizes = [X_train.shape[1]] + hidden_sizes + [2]
    model = MLP(all_layer_sizes)
    optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_dataset = prepare_zipped_XY(X_train, Y_train)
    val_dataset = prepare_zipped_XY(X_val, Y_val)
    train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=my_collate_function)
    val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=my_collate_function)
    print ("Start training!")
    last_epoch_dev_acc = 0
    for epoch in range(train_epochs):
        model.train()
        for batch_X, batch_Y in train_dataloader:
            optim.zero_grad()
            logits = model(batch_X)
            loss = loss_function(logits, batch_Y)
            loss.backward()
            optim.step()
        n_correct, n_total = 0, 0
        model.eval()
        with torch.no_grad():
            for batch_X, batch_Y in val_dataloader:
                logits = model(batch_X)
                predictions = torch.argmax(logits, dim=1)
                n_correct += (predictions == batch_Y).sum().item()
                n_total += batch_Y.size(0)
        last_epoch_dev_acc = n_correct/n_total
        print("Epoch {}, val accuracy {:.2f}".format(epoch+1, last_epoch_dev_acc))
    return last_epoch_dev_acc

train_pytorch_model(X_train, Y_train, X_val, Y_val)

Start training!


C:\Users\manda\AppData\Local\Temp\ipykernel_22868\2790416221.py:24: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:255.)
  return torch.tensor(batch_X).float(), torch.tensor(batch_Y).long()


Epoch 1, val accuracy 0.58
Epoch 2, val accuracy 0.57
Epoch 3, val accuracy 0.58
Epoch 4, val accuracy 0.58
Epoch 5, val accuracy 0.58
Epoch 6, val accuracy 0.58
Epoch 7, val accuracy 0.57
Epoch 8, val accuracy 0.58
Epoch 9, val accuracy 0.57
Epoch 10, val accuracy 0.57


0.5722477064220184

### 4. Hyperparameter tuning (10')
This question requires modifying your previous pytorch training scripts. Use Optuna to find the hyperparameters that can maximize the accuracy on the validation set.  

The range of hyperparameters don't need to be too large (i.e., the total program should still be runnable within a reasonable time). The most important hyperparameter is the learning rate. Other hyperparameters that you can tune include the train epochs, batch size, hidden sizes, etc.  

When you are satisfied with the hyperparameters, report the hyperparameter and the resulting validation accuracy.

In [12]:
# Starter
import optuna 

def train_pytorch_model_with_optuna(trial, X_train, Y_train, X_val, Y_val):
    # TODO -- Modify your train_pytorch_model() in the previous section, so that some hyperparameters are recommended from the Optuna
    train_epochs = trial.suggest_int("train_epochs", 5, 12)
    batch_size = trial.suggest_categorical("batch_size", [64, 128])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    hidden_size = trial.suggest_categorical("hidden_size", [50, 100, 200])
    torch.manual_seed(1)
    model = MLP([X_train.shape[1], hidden_size, 2])
    optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_dataloader = torch.utils.data.DataLoader(prepare_zipped_XY(X_train, Y_train), batch_size=batch_size, shuffle=True, collate_fn=my_collate_function)
    val_dataloader = torch.utils.data.DataLoader(prepare_zipped_XY(X_val, Y_val), batch_size=batch_size, shuffle=False, collate_fn=my_collate_function)
    for epoch in range(train_epochs):
        model.train()
        for batch_X, batch_Y in train_dataloader:
            optim.zero_grad()
            logits = model(batch_X)
            loss = loss_function(logits, batch_Y)
            loss.backward()
            optim.step()
    model.eval()
    n_correct, n_total = 0, 0
    with torch.no_grad():
        for batch_X, batch_Y in val_dataloader:
            predictions = torch.argmax(model(batch_X), dim=1)
            n_correct += (predictions == batch_Y).sum().item()
            n_total += batch_Y.size(0)
    return n_correct / n_total

def find_optimal_hyper_params(X_train, Y_train, X_val, Y_val):
    # Start an Optuna study
    study = optuna.create_study(direction="maximize")

    # TODO -- objective is a function that takes only one argument. Need to also pass in the other arguments. 
    # Hint: You can define another function within the scope of this function
    def objective(trial):
        return train_pytorch_model_with_optuna(trial, X_train, Y_train, X_val, Y_val)
    study.optimize(objective, n_trials=20)
    print("Best hyperparameters:", study.best_params)
    print("Best validation accuracy:", study.best_value)
    return study

find_optimal_hyper_params(X_train, Y_train, X_val, Y_val)

[I 2026-09-16 12:53:21,985] A new study created in memory with name: no-name-e3104a91-30b8-446f-86c6-0346c421203d
[I 2026-09-16 12:55:11,796] Trial 0 finished with value: 0.5745412844036697 and parameters: {'train_epochs': 5, 'batch_size': 64, 'learning_rate': 0.006450065618556958, 'hidden_size': 100}. Best is trial 0 with value: 0.5745412844036697.
[I 2026-09-16 12:58:15,676] Trial 1 finished with value: 0.5825688073394495 and parameters: {'train_epochs': 9, 'batch_size': 128, 'learning_rate': 0.0009696621901546488, 'hidden_size': 200}. Best is trial 1 with value: 0.5825688073394495.
[I 2026-09-16 13:01:35,605] Trial 2 finished with value: 0.5814220183486238 and parameters: {'train_epochs': 10, 'batch_size': 128, 'learning_rate': 0.002725519684527269, 'hidden_size': 200}. Best is trial 1 with value: 0.5825688073394495.
[I 2026-09-16 13:03:29,773] Trial 3 finished with value: 0.5814220183486238 and parameters: {'train_epochs': 5, 'batch_size': 64, 'learning_rate': 0.003936674057800958,

Best hyperparameters: {'train_epochs': 8, 'batch_size': 64, 'learning_rate': 0.0021245687950220676, 'hidden_size': 100}
Best validation accuracy: 0.5871559633027523


### 5. Bonus: Compare the performances of the two methods (2')
Use an appropriate $t$ test, compare the five performance numbers of the sklearn model and the pytorch model *under the same set of hyperparameters*. Do their results differ?

Note: The scores for bonus will be added to the A1 total score, but the total score will be capped to 100%.

In [13]:
# Bonus solution: paired t-test
from scipy.stats import ttest_rel

# Run PyTorch with one fixed set of hyperparameters for all five required seeds.
pytorch_seed_accuracies = []
for seed in [1, 2, 3, 31, 42]:
    torch.manual_seed(seed)
    model = MLP([X_train.shape[1], 100, 2])
    optim = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_function = nn.CrossEntropyLoss()
    train_loader = torch.utils.data.DataLoader(prepare_zipped_XY(X_train, Y_train), batch_size=128, shuffle=True, collate_fn=my_collate_function)
    val_loader = torch.utils.data.DataLoader(prepare_zipped_XY(X_val, Y_val), batch_size=128, shuffle=False, collate_fn=my_collate_function)
    for epoch in range(10):
        model.train()
        for batch_X, batch_Y in train_loader:
            optim.zero_grad()
            loss = loss_function(model(batch_X), batch_Y)
            loss.backward()
            optim.step()
    model.eval(); n_correct = 0; n_total = 0
    with torch.no_grad():
        for batch_X, batch_Y in val_loader:
            pred = torch.argmax(model(batch_X), dim=1)
            n_correct += (pred == batch_Y).sum().item(); n_total += batch_Y.size(0)
    pytorch_seed_accuracies.append(n_correct / n_total)

t_stat, p_value = ttest_rel(sklearn_seed_accuracies, pytorch_seed_accuracies)
print("Sklearn scores:", sklearn_seed_accuracies)
print("PyTorch scores:", pytorch_seed_accuracies)
print("Paired t-statistic:", t_stat)
print("p-value:", p_value)
if p_value < 0.05:
    print("At alpha=0.05, there is evidence of a statistically significant difference.")
else:
    print("At alpha=0.05, there is not sufficient evidence of a statistically significant difference.")

Sklearn scores: [0.5814220183486238, 0.5779816513761468, 0.5791284403669725, 0.5837155963302753, 0.5860091743119266]
PyTorch scores: [0.5711009174311926, 0.5825688073394495, 0.5779816513761468, 0.5768348623853211, 0.5745412844036697]
Paired t-statistic: 1.6799278063066811
p-value: 0.16826913872781
At alpha=0.05, there is not sufficient evidence of a statistically significant difference.
